# Notebook 2 — Limpieza y Feature Engineering Secuencial

**Objetivo**: cruzar el histórico de partidos con el histórico de Elo y construir el
tablón de features, evitando a toda costa la fuga de información temporal (*data
leakage*): cada feature de un partido se calcula **solo** con información estrictamente
anterior a su fecha.

Entrada: `data/raw/results.csv`, `data/raw/elo_historico.csv` (Notebook 1).
Salida: `data/processed/partidos_features.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = DIR_RAIZ / "data" / "raw"
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

VENTANAS_ROLLING = (5, 10)   # partidos previos sobre los que promediar forma
DIAS_TENDENCIA_ELO = 180     # ~6 meses, para el feature de tendencia de Elo

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

df_partidos = pd.read_csv(DIR_RAW / "results.csv", parse_dates=["date"])
df_elo = pd.read_csv(DIR_RAW / "elo_historico.csv", parse_dates=["fecha"])

print(f"Partidos: {len(df_partidos):,}  |  Filas de Elo: {len(df_elo):,}")

Partidos: 49,499  |  Filas de Elo: 36,099


## 2.1 Limpieza básica

Normalización de nombres de columna, tipado y una regla de negocio explícita: un
partido sin marcador (programado pero aún no jugado — el caso de las eliminatorias
del Mundial 2026 que siguen en juego) se conserva pero se marca aparte, porque **no
debe entrar como fila de entrenamiento**, aunque sí puede ser el objeto de una
predicción más adelante.

In [2]:
def limpiar_partidos(df: pd.DataFrame) -> pd.DataFrame:
    """Tipa y normaliza el histórico crudo; no elimina partidos sin marcador
    (los necesita el Notebook 4 para saber qué predecir), solo los señala.
    """
    df = df.rename(columns={
        "date": "fecha", "home_team": "equipo_local", "away_team": "equipo_visitante",
        "home_score": "goles_local", "away_score": "goles_visitante",
        "tournament": "torneo", "neutral": "es_neutral",
    }).copy()

    df["fecha"] = pd.to_datetime(df["fecha"])
    df["jugado"] = df["goles_local"].notna() & df["goles_visitante"].notna()
    df["es_neutral"] = df["es_neutral"].astype(bool)

    # Duplicados por (día, equipos): distintas fuentes pueden registrar el
    # mismo partido con una hora ligeramente distinta — comparar solo el día
    # evita contarlo dos veces en las ventanas móviles de más abajo.
    df["_dia"] = df["fecha"].dt.normalize()
    antes = len(df)
    df = df.drop_duplicates(subset=["_dia", "equipo_local", "equipo_visitante"]).drop(columns="_dia")
    if antes - len(df):
        print(f"Eliminados {antes - len(df)} partidos duplicados (mismo día y equipos)")

    return df.sort_values("fecha").reset_index(drop=True)


df_partidos = limpiar_partidos(df_partidos)
print(f"Partidos jugados: {df_partidos['jugado'].sum():,} / {len(df_partidos):,} totales")
df_partidos.tail()

Eliminados 2 partidos duplicados (mismo día y equipos)
Partidos jugados: 49,488 / 49,497 totales


,fecha,equipo_local,equipo_visitante,goles_local,goles_visitante,torneo,city,country,es_neutral,jugado
49492,2026-07-04,Canada,Morocco,NaN,NaN,FIFA World Cup,Houston,United States,True,False
49493,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,False
49494,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,False
49495,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,Dallas,United States,True,False
49496,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,Seattle,United States,False,False


## 2.2 Formato largo (una fila = un equipo en un partido)

Las ventanas móviles ("¿cómo le fueron a este equipo sus últimos N partidos?") son
triviales en formato largo y un infierno en formato ancho (local/visitante como
columnas separadas). Se paga la duplicación de memoria aquí a cambio de no mantener
dos ramas de código idénticas para cada feature.

In [3]:
def a_formato_largo(df: pd.DataFrame) -> pd.DataFrame:
    """Duplica cada partido en dos filas, una por equipo, con 'gf'/'gc'
    (goles a favor / en contra) desde la perspectiva de ESE equipo.
    """
    base = ["fecha", "torneo", "jugado"]
    local = df[base + ["equipo_local", "goles_local", "goles_visitante"]].rename(
        columns={"equipo_local": "equipo", "goles_local": "gf", "goles_visitante": "gc"}
    )
    local["es_local"] = True
    visitante = df[base + ["equipo_visitante", "goles_visitante", "goles_local"]].rename(
        columns={"equipo_visitante": "equipo", "goles_visitante": "gf", "goles_local": "gc"}
    )
    visitante["es_local"] = False

    largo = pd.concat([local, visitante], ignore_index=True)
    largo["id_partido"] = pd.concat([df.index.to_series(), df.index.to_series()], ignore_index=True)
    largo = largo.sort_values(["equipo", "fecha"]).reset_index(drop=True)

    # Puntos solo tienen sentido para partidos ya jugados.
    largo["puntos"] = np.select(
        [largo["jugado"] & (largo.gf > largo.gc), largo["jugado"] & (largo.gf == largo.gc)],
        [3, 1], default=np.where(largo["jugado"], 0, np.nan),
    )
    return largo


df_largo = a_formato_largo(df_partidos)
df_largo.head()

,fecha,torneo,jugado,equipo,gf,gc,es_local,id_partido,puntos
0,2012-09-25,Friendly,True,Abkhazia,1.0,1.0,True,36255,1.0
1,2012-10-21,Friendly,True,Abkhazia,0.0,3.0,False,36402,0.0
2,2013-09-23,Friendly,True,Abkhazia,3.0,0.0,True,37316,3.0
3,2014-06-01,CONIFA World Football Cup,True,Abkhazia,1.0,1.0,True,37787,1.0
4,2014-06-02,CONIFA World Football Cup,True,Abkhazia,2.0,1.0,False,37793,3.0


## 2.3 Elo exacto por fecha (sin fuga)

Cada fila del histórico de Elo trae la puntuación **posterior** al partido de esa
selección. Para asignar a un partido el Elo que un equipo tenía **antes** de jugarlo
—incluido el caso en que la fecha coincide exactamente con una fila de su propio
historial de Elo, que es precisamente el caso de los partidos que se quieren predecir—
se calcula primero `elo_antes = elo.shift(1)` dentro de cada selección, y **sobre esa
serie ya desplazada** se hace el cruce por fecha (`merge_asof` hacia atrás). Sin el
`shift(1)` previo, un partido tomaría su propio Elo posterior como si fuera el de
entrada: la fuga de información más sutil (y más grave) de todo el pipeline.

In [4]:
def calcular_elo_antes_partido(df_elo: pd.DataFrame) -> pd.DataFrame:
    """Añade 'elo_antes': el Elo de la selección ENTRANDO a cada partido suyo."""
    df_elo = df_elo.sort_values(["equipo", "fecha"]).copy()
    df_elo["elo_antes"] = df_elo.groupby("equipo")["elo"].shift(1)
    # Antes del primer partido registrado de una selección: 1500 es el punto
    # de partida estándar de cualquier sistema Elo (ausencia total de
    # información previa), no un valor inventado ad hoc.
    df_elo["elo_antes"] = df_elo["elo_antes"].fillna(1500.0)
    return df_elo[["equipo", "fecha", "elo_antes"]]


def asignar_elo_por_fecha(df_largo: pd.DataFrame, df_elo_antes: pd.DataFrame,
                           dias_atras: int = 0, nombre_columna: str = "elo") -> pd.Series:
    """Cruza cada fila de `df_largo` con el Elo de ESE equipo vigente en su fecha
    (o `dias_atras` antes, para el feature de tendencia). `merge_asof` exige
    ambos lados ordenados por la columna de cruce.
    """
    izquierda = df_largo[["equipo", "fecha"]].copy()
    izquierda["fecha_consulta"] = izquierda["fecha"] - pd.Timedelta(days=dias_atras)
    izquierda = izquierda.reset_index().sort_values("fecha_consulta")

    cruce = pd.merge_asof(
        izquierda, df_elo_antes.sort_values("fecha"),
        left_on="fecha_consulta", right_on="fecha", by="equipo", direction="backward",
    ).set_index("index").sort_index()

    return cruce["elo_antes"].rename(nombre_columna)


df_elo_antes = calcular_elo_antes_partido(df_elo)

df_largo["elo_actual"] = asignar_elo_por_fecha(df_largo, df_elo_antes)
df_largo["elo_hace_180d"] = asignar_elo_por_fecha(df_largo, df_elo_antes, dias_atras=DIAS_TENDENCIA_ELO)
# Selecciones fuera del histórico de Elo (no forman parte del Mundial 2026):
# 1500 neutro, igual que un equipo sin historial — no invalida el resto de
# features de ese partido, solo dice "no tenemos información de Elo aquí".
df_largo[["elo_actual", "elo_hace_180d"]] = df_largo[["elo_actual", "elo_hace_180d"]].fillna(1500.0)
df_largo["tendencia_elo"] = df_largo["elo_actual"] - df_largo["elo_hace_180d"]

df_largo[["equipo", "fecha", "elo_actual", "elo_hace_180d", "tendencia_elo"]].tail()

,equipo,fecha,elo_actual,elo_hace_180d,tendencia_elo
98989,Åland Islands,2017-06-29,1500.0,1500.0,0.0
98990,Åland Islands,2023-07-09,1500.0,1500.0,0.0
98991,Åland Islands,2023-07-10,1500.0,1500.0,0.0
98992,Åland Islands,2023-07-11,1500.0,1500.0,0.0
98993,Åland Islands,2023-07-13,1500.0,1500.0,0.0


## 2.4 Ventanas móviles (forma, racha, descanso)

Para cada ventana de `VENTANAS_ROLLING` (5 y 10 partidos): promedio de goles a favor
y en contra, y racha de puntos. Siempre `shift(1)` antes del `rolling`: la fila del
partido en curso nunca puede ver su propio resultado.

In [5]:
def calcular_features_forma(df_largo: pd.DataFrame, ventanas: tuple[int, ...]) -> pd.DataFrame:
    """Añade, por cada ventana N: prom_gf_N, prom_gc_N (forma ofensiva/defensiva)
    y racha_puntos_N (suma de puntos de los N partidos anteriores).
    """
    df_largo = df_largo.sort_values(["equipo", "fecha"]).copy()
    g = df_largo.groupby("equipo", sort=False)

    for ventana in ventanas:
        for col, nombre in (("gf", "prom_gf"), ("gc", "prom_gc")):
            df_largo[f"{nombre}_{ventana}"] = g[col].transform(
                lambda s, v=ventana: s.shift(1).rolling(v, min_periods=1).mean()
            )
        df_largo[f"racha_puntos_{ventana}"] = g["puntos"].transform(
            lambda s, v=ventana: s.shift(1).rolling(v, min_periods=1).sum()
        )

    # Un equipo sin historial previo arranca en un punto neutro: balance de
    # goles en cero y racha equivalente a un punto por partido (ni dominante
    # ni en crisis), nunca NaN (contaminaría el entrenamiento aguas abajo).
    for ventana in ventanas:
        for col in (f"prom_gf_{ventana}", f"prom_gc_{ventana}"):
            df_largo[col] = df_largo[col].fillna(1.0)
        df_largo[f"racha_puntos_{ventana}"] = df_largo[f"racha_puntos_{ventana}"].fillna(float(ventana))

    return df_largo


def calcular_dias_descanso(df_largo: pd.DataFrame) -> pd.DataFrame:
    """Días desde el partido anterior de la misma selección — la fatiga
    acumulada / falta de ritmo competitivo es una variable de forma clásica
    que las métricas puramente ofensivas/defensivas no capturan.
    """
    df_largo = df_largo.sort_values(["equipo", "fecha"]).copy()
    fecha_anterior = df_largo.groupby("equipo")["fecha"].shift(1)
    df_largo["dias_descanso"] = (df_largo["fecha"] - fecha_anterior).dt.days
    # Primer partido conocido de una selección: se asume un descanso "normal"
    # (la mediana real del propio dataset), no un cero que sugeriría fatiga.
    df_largo["dias_descanso"] = df_largo["dias_descanso"].fillna(df_largo["dias_descanso"].median())
    return df_largo


df_largo = calcular_features_forma(df_largo, VENTANAS_ROLLING)
df_largo = calcular_dias_descanso(df_largo)
df_largo.filter(regex="equipo|fecha|prom_gf|racha_puntos_5|dias_descanso").tail()

,fecha,equipo,prom_gf_5,racha_puntos_5,prom_gf_10,dias_descanso
98989,2017-06-29,Åland Islands,0.8,6.0,1.1,2.0
98990,2023-07-09,Åland Islands,1.0,9.0,1.1,2201.0
98991,2023-07-10,Åland Islands,1.0,8.0,0.9,1.0
98992,2023-07-11,Åland Islands,1.0,7.0,0.9,1.0
98993,2023-07-13,Åland Islands,1.0,6.0,0.7,2.0


## 2.5 Volver a formato ancho y ensamblar el tablón final

Cada partido necesita las features de AMBOS lados en la misma fila (sufijo `_local` /
`_visitante`), más las diferencias directas que de verdad usará el modelo (`elo_diff`,
diferencias de forma).

In [6]:
COLUMNAS_FEATURE = [
    "elo_actual", "tendencia_elo", "dias_descanso",
    *[f"prom_gf_{v}" for v in VENTANAS_ROLLING],
    *[f"prom_gc_{v}" for v in VENTANAS_ROLLING],
    *[f"racha_puntos_{v}" for v in VENTANAS_ROLLING],
]


def ensamblar_tablon_final(df_partidos: pd.DataFrame, df_largo: pd.DataFrame) -> pd.DataFrame:
    """Unifica de vuelta a formato ancho: una fila por partido, con las
    features de ambos equipos y las diferencias derivadas.
    """
    mitad_local = df_largo[df_largo["es_local"]].set_index("id_partido")
    mitad_visitante = df_largo[~df_largo["es_local"]].set_index("id_partido")

    tablon = df_partidos.copy()
    for col in COLUMNAS_FEATURE:
        tablon[f"{col}_local"] = tablon.index.map(mitad_local[col])
        tablon[f"{col}_visitante"] = tablon.index.map(mitad_visitante[col])

    tablon["elo_diff"] = tablon["elo_actual_local"] - tablon["elo_actual_visitante"]
    for v in VENTANAS_ROLLING:
        tablon[f"dif_forma_gf_{v}"] = tablon[f"prom_gf_{v}_local"] - tablon[f"prom_gf_{v}_visitante"]
        tablon[f"dif_racha_{v}"] = tablon[f"racha_puntos_{v}_local"] - tablon[f"racha_puntos_{v}_visitante"]

    # Resultado 1X2, solo para partidos ya jugados (target de clasificación).
    tablon["resultado_1x2"] = np.select(
        [tablon["jugado"] & (tablon.goles_local > tablon.goles_visitante),
         tablon["jugado"] & (tablon.goles_local == tablon.goles_visitante)],
        ["LOCAL", "EMPATE"], default=np.where(tablon["jugado"], "VISITANTE", None),
    )
    return tablon


df_features = ensamblar_tablon_final(df_partidos, df_largo)
columnas_features_finales = [c for c in df_features.columns if c not in df_partidos.columns]
print(f"Tablón final: {df_features.shape}")
print(f"NaN en columnas de features (entre partidos jugados): "
      f"{df_features.loc[df_features['jugado'], columnas_features_finales].isna().sum().sum()}")
df_features[df_features["jugado"]].tail()[["fecha", "equipo_local", "equipo_visitante",
                                            "goles_local", "goles_visitante",
                                            "elo_diff", "dif_forma_gf_5", "resultado_1x2"]]

Tablón final: (49497, 34)
NaN en columnas de features (entre partidos jugados): 0


,fecha,equipo_local,equipo_visitante,goles_local,goles_visitante,elo_diff,dif_forma_gf_5,resultado_1x2
49483,2026-07-01,United States,Bosnia and Herzegovina,2.0,0.0,159.0,1.2,LOCAL
49484,2026-07-01,Belgium,Senegal,2.0,2.0,42.0,0.6,EMPATE
49485,2026-07-02,Spain,Austria,3.0,0.0,308.0,0.2,LOCAL
49486,2026-07-02,Portugal,Croatia,2.0,1.0,85.0,0.6,LOCAL
49487,2026-07-02,Switzerland,Algeria,2.0,0.0,129.0,0.4,LOCAL


## 2.6 Persistencia en `data/processed/`

In [7]:
ruta_salida = DIR_PROCESSED / "partidos_features.csv"
df_features.to_csv(ruta_salida, index=False)
print(f"Guardado: {ruta_salida} ({len(df_features):,} filas, {df_features.shape[1]} columnas)")

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/processed/partidos_features.csv (49,497 filas, 34 columnas)


## Resumen

El tablón `data/processed/partidos_features.csv` tiene una fila por partido (jugado o
no) con:
- Elo de ambos equipos **antes** del partido, su diferencia y su tendencia a 6 meses.
- Forma ofensiva/defensiva y racha de puntos a 5 y 10 partidos, con `shift(1)` en
  todos los casos.
- Días de descanso desde el partido anterior.

Ninguna de estas columnas usa información posterior a la fecha del partido que describe.

Siguiente paso: **Notebook 3**, el análisis exploratorio que valida los supuestos
matemáticos (Poisson, estacionaridad, colinealidad) antes de modelar.